In [21]:
import os
os.chdir("/home/storm/Assignment/Assignment1")
print(os.getcwd())
!ls


/home/storm/Assignment/Assignment1
YourName.assignment-1.ipynb    mypos.train.txt	      test.tag.out
clean_output.txt	       otest.1k.nopipe.txt    test1.tagger.txt
convert_mypos.py	       pos_tag.model	      test2.crfsuite.txt
mk-wordtag.pl		       random_output.txt      test2.tagger.txt
myWordTagger.py		       ref_hyp.out.txt	      train.crfsuite.txt
mypos-ver.3.0.shuf.nopipe.txt  test.crfsuite.out.txt
mypos.test.txt		       test.crfsuite.txt


In [22]:
!which crfsuite
!crfsuite --help | head -n 15
!wget -nc https://github.com/ye-kyaw-thu/myPOS/raw/refs/heads/master/corpus-ver-3.0/corpus/mypos-ver.3.0.shuf.nopipe.txt
!wget -nc https://raw.githubusercontent.com/ye-kyaw-thu/myPOS/refs/heads/master/corpus-ver-3.0/corpus/otest.1k.nopipe.txt
!wc -l mypos-ver.3.0.shuf.nopipe.txt otest.1k.nopipe.txt
!head -n 3 mypos-ver.3.0.shuf.nopipe.txt
!head -n 3 otest.1k.nopipe.txt


/usr/local/bin/crfsuite
CRFSuite 0.12.2  Copyright (c) 2007-2013 Naoaki Okazaki

USAGE: crfsuite <COMMAND> [OPTIONS]
    COMMAND     Command name to specify the processing
    OPTIONS     Arguments for the command (optional; command-specific)

COMMAND:
    learn       Obtain a model from a training set of instances
    tag         Assign suitable labels to given instances by using a model
    dump        Output a model in a plain-text format

For the usage of each command, specify -h option in the command argument.
File ‘mypos-ver.3.0.shuf.nopipe.txt’ already there; not retrieving.

File ‘otest.1k.nopipe.txt’ already there; not retrieving.

  43196 mypos-ver.3.0.shuf.nopipe.txt
   1000 otest.1k.nopipe.txt
  44196 total
၁၉၆၂/num ခုနှစ်/n ခန့်မှန်း/v သန်းခေါင်စာရင်း/n အရ/ppm လူဦးရေ/n ၁၁၅၉၃၁/num ယောက်/part ရှိ/v သည်/ppm ။/punc
လူ/n တိုင်း/part တွင်/ppm သင့်မြတ်/v လျော်ကန်/v စွာ/part ကန့်သတ်/v ထား/part သည့်/part အလုပ်/n လုပ်/v ချိန်/n အပြင်/conj ၊/punc လစာ/n နှင့်တကွ/conj အခါ/n ကာလ/n အားလျ

In [23]:
pwd

'/home/storm/Assignment/Assignment1'

In [24]:
%%writefile convert_mypos.py
def convert_mypos(input_file, output_file):
    with open(input_file, encoding="utf-8") as f:
        lines = f.readlines()
    with open(output_file, "w", encoding="utf-8") as out:
        for line in lines:
            tokens = line.strip().split()
            for token in tokens:
                if "/" not in token:
                    continue
                word, tag = token.rsplit("/", 1)
                out.write(f"{word} {tag}\n")
            out.write("\n")

if __name__ == "__main__":
    convert_mypos("mypos-ver.3.0.shuf.nopipe.txt", "mypos.train.txt")
    convert_mypos("otest.1k.nopipe.txt", "mypos.test.txt")
    print("Done")

Overwriting convert_mypos.py


In [25]:
!python3 convert_mypos.py

Done


In [26]:
!head -n 20 mypos.train.txt
!head -n 20 mypos.test.txt

၁၉၆၂ num
ခုနှစ် n
ခန့်မှန်း v
သန်းခေါင်စာရင်း n
အရ ppm
လူဦးရေ n
၁၁၅၉၃၁ num
ယောက် part
ရှိ v
သည် ppm
။ punc

လူ n
တိုင်း part
တွင် ppm
သင့်မြတ် v
လျော်ကန် v
စွာ part
ကန့်သတ် v
ထား part
တစ် tn
ကိုက် n
ကို ppm
ဝမ် n
ခုနှစ်ထောင် tn
ပါ part
။ punc

မနှစ် n
က ppm
သူ pron
ကျွန်မ pron
ကို ppm
သင် v
ပေး part
တယ် ppm
။ punc

ကျွန်တော့် pron
ခုံ n


In [27]:
%%writefile /home/storm/crfsuite/example/mypos_features.py
import crfutils

fields = 'w y'
separator = ' '

templates = (
    (('w', -2), ),
    (('w', -1), ),
    (('w',  0), ),
    (('w',  1), ),
    (('w',  2), ),
    (('w', -1), ('w', 0)),
    (('w',  0), ('w', 1)),
)

def feature_extractor(X):
    crfutils.apply_templates(X, templates)
    if X:
        X[0]['F'].append('__BOS__')
        X[-1]['F'].append('__EOS__')

if __name__ == '__main__':
    crfutils.main(feature_extractor, fields=fields, sep=separator)

Overwriting /home/storm/crfsuite/example/mypos_features.py


In [28]:
!cat mypos.train.txt | python3 /home/storm/crfsuite/example/mypos_features.py > train.crfsuite.txt
!cat mypos.test.txt  | python3 /home/storm/crfsuite/example/mypos_features.py > test.crfsuite.txt
!head -n 25 train.crfsuite.txt
!wc -l train.crfsuite.txt test.crfsuite.txt


num	w[0]=၁၉၆၂	w[1]=ခုနှစ်	w[2]=ခန့်မှန်း	w[0]|w[1]=၁၉၆၂|ခုနှစ်	__BOS__
n	w[-1]=၁၉၆၂	w[0]=ခုနှစ်	w[1]=ခန့်မှန်း	w[2]=သန်းခေါင်စာရင်း	w[-1]|w[0]=၁၉၆၂|ခုနှစ်	w[0]|w[1]=ခုနှစ်|ခန့်မှန်း
v	w[-2]=၁၉၆၂	w[-1]=ခုနှစ်	w[0]=ခန့်မှန်း	w[1]=သန်းခေါင်စာရင်း	w[2]=အရ	w[-1]|w[0]=ခုနှစ်|ခန့်မှန်း	w[0]|w[1]=ခန့်မှန်း|သန်းခေါင်စာရင်း
n	w[-2]=ခုနှစ်	w[-1]=ခန့်မှန်း	w[0]=သန်းခေါင်စာရင်း	w[1]=အရ	w[2]=လူဦးရေ	w[-1]|w[0]=ခန့်မှန်း|သန်းခေါင်စာရင်း	w[0]|w[1]=သန်းခေါင်စာရင်း|အရ
ppm	w[-2]=ခန့်မှန်း	w[-1]=သန်းခေါင်စာရင်း	w[0]=အရ	w[1]=လူဦးရေ	w[2]=၁၁၅၉၃၁	w[-1]|w[0]=သန်းခေါင်စာရင်း|အရ	w[0]|w[1]=အရ|လူဦးရေ
n	w[-2]=သန်းခေါင်စာရင်း	w[-1]=အရ	w[0]=လူဦးရေ	w[1]=၁၁၅၉၃၁	w[2]=ယောက်	w[-1]|w[0]=အရ|လူဦးရေ	w[0]|w[1]=လူဦးရေ|၁၁၅၉၃၁
num	w[-2]=အရ	w[-1]=လူဦးရေ	w[0]=၁၁၅၉၃၁	w[1]=ယောက်	w[2]=ရှိ	w[-1]|w[0]=လူဦးရေ|၁၁၅၉၃၁	w[0]|w[1]=၁၁၅၉၃၁|ယောက်
part	w[-2]=လူဦးရေ	w[-1]=၁၁၅၉၃၁	w[0]=ယောက်	w[1]=ရှိ	w[2]=သည်	w[-1]|w[0]=၁၁၅၉၃၁|ယောက်	w[0]|w[1]=ယောက်|ရှိ
v	w[-2]=၁၁၅၉၃၁	w[-1]=ယောက်	w[0]=ရှိ	w[1]=သည်	w[2]=။	w[-1]|w[0]=ယောက်|ရှိ	w[0]|w[1]=ရှိ|သည်
ppm	w[-2

In [29]:
!time crfsuite learn -m pos_tag.model train.crfsuite.txt
!ls -lh pos_tag.model

CRFSuite 0.12.2  Copyright (c) 2007-2013 Naoaki Okazaki

Start time of the training: 2026-07-31T18:15:09Z

Reading the data set(s)
[1] train.crfsuite.txt
0....1....2....3....4....5....6....7....8....9....10
Number of instances: 43197
Seconds required: 2.521

Statistics the data set(s)
Number of data sets (groups): 1
Number of instances: 43196
Number of items: 564517
Number of attributes: 452474
Number of labels: 15

Feature generation
type: CRF1d
feature.minfreq: 0.000000
feature.possible_states: 0
feature.possible_transitions: 0
0....1....2....3....4....5....6....7....8....9....10
Number of features: 571865
Seconds required: 0.962

L-BFGS optimization
c1: 0.000000
c2: 1.000000
num_memories: 6
max_iterations: 2147483647
epsilon: 0.000010
stop: 10
delta: 0.000010
linesearch: MoreThuente
linesearch.max_iterations: 20

***** Iteration #1 *****
Loss: 963962.209494
Feature norm: 5.000000
Error norm: 77493.052921
Active features: 571865
Line search trials: 2
Line search step: 0.000036
Second

In [30]:
!time crfsuite tag -m pos_tag.model test.crfsuite.txt > test.tag.out
!head -n 40 test.tag.out


real	0m0.060s
user	0m0.032s
sys	0m0.028s
tn
n
ppm
n
num
part
punc

n
ppm
pron
pron
ppm
v
part
ppm
punc

pron
n
v
v
part
punc

n
v
part
v
ppm
part
punc

n
adv
v
part
punc
tn
tn


In [31]:
!time crfsuite tag -r -m pos_tag.model test.crfsuite.txt > ref_hyp.out.txt
!head -n 50 ref_hyp.out.txt


real	0m0.062s
user	0m0.032s
sys	0m0.028s
tn	tn
n	n
ppm	ppm
n	n
tn	num
part	part
punc	punc

n	n
ppm	ppm
pron	pron
pron	pron
ppm	ppm
v	v
part	part
ppm	ppm
punc	punc

pron	pron
n	n
v	v
v	v
part	part
punc	punc

n	n
v	v
part	part
v	v
ppm	ppm
part	part
punc	punc

n	n
adv	adv
v	v
part	part
punc	punc
tn	tn
tn	tn
n	n
part	part
v	v
part	part
conj	conj
v	v
part	part
ppm	ppm
part	part
punc	punc


In [32]:
!time crfsuite tag -qt -m pos_tag.model test.crfsuite.txt


Performance by label (#match, #model, #ref) (precision, recall, F1):
    num: (146, 147, 155) (0.9932, 0.9419, 0.9669)
    n: (2963, 3102, 3000) (0.9552, 0.9877, 0.9712)
    v: (1948, 2007, 2010) (0.9706, 0.9692, 0.9699)
    ppm: (2035, 2063, 2060) (0.9864, 0.9879, 0.9871)
    part: (3134, 3185, 3189) (0.9840, 0.9828, 0.9834)
    punc: (1270, 1270, 1270) (1.0000, 1.0000, 1.0000)
    conj: (394, 415, 411) (0.9494, 0.9586, 0.9540)
    adj: (324, 345, 366) (0.9391, 0.8852, 0.9114)
    adv: (233, 246, 262) (0.9472, 0.8893, 0.9173)
    pron: (459, 469, 476) (0.9787, 0.9643, 0.9714)
    tn: (138, 138, 142) (1.0000, 0.9718, 0.9857)
    fw: (46, 46, 87) (1.0000, 0.5287, 0.6917)
    int: (23, 23, 25) (1.0000, 0.9200, 0.9583)
    sb: (3, 3, 3) (1.0000, 1.0000, 1.0000)
    abb: (9, 9, 12) (1.0000, 0.7500, 0.8571)
Macro-average precision, recall, F1: (0.980251, 0.915828, 0.941700)
Item accuracy: 13125 / 13468 (0.9745)
Instance accuracy: 747 / 1000 (0.7470)
Elapsed time: 0.028736 [sec] (34799.6 [in